In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1: LOGGING AND IMPORTS - MODIFY THESE PATHS AS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════
#jupyter cell test notebook for pipeline integration 
# DO NOT OVERWRITE LOGS, CREATE NEW LOG FILE BEFORE EACH NEW RUN WITH TIME AND DATE 
import os
import re
import io
import sys
import json
import ujson
import ast
import time
import glob
from pathlib import Path
import logging

# ═══════════════════════════════════════════════════════════════════════════════
# AUTOMATIC PATH SETUP - NO NEED TO MODIFY UNLESS YOU WANT CUSTOM PATHS
# ═══════════════════════════════════════════════════════════════════════════════

# Get the directory where this notebook is located
NOTEBOOK_DIR = Path.cwd()

# Set up paths relative to notebook location
LOG_DIR = NOTEBOOK_DIR / "logs"
PDF_INPUT_FOLDER = NOTEBOOK_DIR / "sample_pdfs"
OUTPUT_FOLDER = NOTEBOOK_DIR / "output"

# Create directories if they don't exist
LOG_DIR.mkdir(exist_ok=True)
PDF_INPUT_FOLDER.mkdir(exist_ok=True)
OUTPUT_FOLDER.mkdir(exist_ok=True)

# Convert to strings for compatibility
LOG_DIR = str(LOG_DIR)
PDF_INPUT_FOLDER = str(PDF_INPUT_FOLDER)
# OUTPUT_FEATHER = str(OUTPUT_FOLDER / "results_removed_invisible_text.feather") 2/25/2026
OUTPUT_FEATHER = str(OUTPUT_FOLDER / "3_15_2026_test_2.feather") # 2/25/2026

# LOG_FILE = os.path.join(LOG_DIR, "invisible_text_fixed.log") 2/25/2026
LOG_FILE = os.path.join(LOG_DIR, "3_15_2026_test_2.log") # 2/25/2026

# CLEAR EXISTING HANDLERS FIRST
root_logger = logging.getLogger()
for handler in root_logger.handlers[:]:
    root_logger.removeHandler(handler)

# Now configure logging
logging.basicConfig(
    filename=LOG_FILE,
    filemode='w',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True  # Python 3.8+ - forces reconfiguration
)

# Also add console handler so you see logs in notebook
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
root_logger.addHandler(console_handler)

# Set loggers for docling modules
docling_logger = logging.getLogger("docling")
docling_logger.setLevel(logging.DEBUG)
docling_logger.propagate = True

_log = logging.getLogger(__name__)
_log.setLevel(logging.DEBUG)

_log.debug("Test debug message from LayoutPostprocessor.py")
_log.info("Test info message from LayoutPostprocessor.py")

def print_and_log(message, level="debug"):
    """Print to console and log at the same time"""
    print(message)
    if level == "info":
        _log.info(message)
    elif level == "debug":
        _log.debug(message)
    elif level == "error":
        _log.error(message)
    elif level == "warning":
        _log.warning(message)

import traceback  # [QA CHANGE] For logging full tracebacks
import requests
import pandas as pd

import tiktoken
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

print("="*80)
print("SIMPLE PDF PROCESSOR SETUP")
print("="*80)
print(f"📁 Put your PDF files in: {PDF_INPUT_FOLDER}")
print(f"📄 Results will be saved to: {OUTPUT_FEATHER}")
print(f"📋 Logs will be saved to: {LOG_DIR}")
print("="*80)

# Docker run command 

for code formula extraction (code_formula_model.py sends to port 8006). if you dont have this container set up code formula will not work

```
docker run --name docling-granite-vision \
  --gpus '"device=0,3"' \
  --privileged \
  --ipc=host \
  -p 8006:8000 \
  -e OMP_NUM_THREADS=6 \
  -e VLLM_USE_V1=1 \
  -e CUDA_DEVICE_ORDER=PCI_BUS_ID \
  -e CUDA_VISIBLE_DEVICES=0,3 \
  -v /mnt/c/Users/WSTATION/Desktop/docling_mods/model_cache:/root/.cache/huggingface \
  nvcr.io/nvidia/vllm:25.09-py3 \
  vllm serve ibm-granite/granite-vision-3.3-2b \
      --port 8000 \
	  --tensor-parallel-size 2 \
	  --gpu-memory-utilization 0.9 \
	  --trust-remote-code \
	  --max-model-len 16384 
      --limit-mm-per-prompt '{"image": 1}' \
      --dtype auto
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2: CONFIGURATION - PATHS ARE NOW AUTOMATIC
# ═══════════════════════════════════════════════════════════════════════════════

print_and_log(f"PDF Input Folder: {PDF_INPUT_FOLDER}")
print_and_log(f"Output File: {OUTPUT_FEATHER}")
print_and_log(f"Log Directory: {LOG_DIR}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3: IMPORT DOCLING PROCESSING FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

try:
    from docling_extract_formulas_mp_multi_provenance import do_docling_extraction
    print_and_log("[+] Successfully imported docling extraction function")
except ImportError as e:
    print_and_log(f"[!] Could not import docling extraction function: {e}", "error")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4: PDF DISCOVERY FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

def create_pdf_dataframe(pdf_folder):
    """
    Scan a folder for PDF files and create a minimal DataFrame for processing.
    Args:
        pdf_folder (str): Path to folder containing PDF files
    Returns:
        pd.DataFrame: DataFrame with PDFPath column
    """
    # Check if folder exists
    if not os.path.exists(pdf_folder):
        raise FileNotFoundError(f"PDF folder not found: {pdf_folder}")
    
    # Find all PDF files in the folder
    pdf_pattern = os.path.join(pdf_folder, "*.pdf")
    pdf_files = glob.glob(pdf_pattern)
    
    if not pdf_files:
        raise ValueError(f"No PDF files found in {pdf_folder}")
    
    print_and_log(f"[+] Found {len(pdf_files)} PDF files:")
    for pdf_file in pdf_files:
        print_and_log(f"    - {os.path.basename(pdf_file)}")
    
    # Create DataFrame - just PDFPath column
    df = pd.DataFrame({
        "PDFPath": pdf_files,
        "FileName": [os.path.basename(path) for path in pdf_files]
    })
    
    print_and_log(f"Created DataFrame with {len(df)} PDF files")
    
    return df

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5: CREATE DATAFRAME FROM YOUR PDF FOLDER
# ═══════════════════════════════════════════════════════════════════════════════

print_and_log(f"\nScanning for PDFs in: {PDF_INPUT_FOLDER}")

# Check if input folder exists
if not os.path.exists(PDF_INPUT_FOLDER):
    print_and_log(f"[!] Input folder does not exist: {PDF_INPUT_FOLDER}", "error")
    print_and_log("Please add PDF files to the 'sample_pdfs' folder", "error")
else:
    try:
        pdf_df = create_pdf_dataframe(PDF_INPUT_FOLDER)
        print_and_log(f"[+] Created DataFrame with {len(pdf_df)} files")
        
        # Display the DataFrame
        print_and_log("\nDataFrame contents:")
        print(pdf_df)
    except ValueError as e:
        print_and_log(f"[!] {e}", "error")
        print_and_log("Please add some PDF files to the 'sample_pdfs' folder", "error")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6: PROCESS PDFs WITH DOCLING
# ═══════════════════════════════════════════════════════════════════════════════

if 'pdf_df' in locals() and len(pdf_df) > 0:
    print_and_log(f"Processing {len(pdf_df)} PDFs with Docling...")
    print_and_log("This may take several minutes depending on PDF size and complexity...")
    
    try:
        processed_df = do_docling_extraction(pdf_df, output_dir="/mnt/c/Users/WSTATION/Desktop/docling_mods/docling_debug")
        print_and_log(f"[+] Successfully processed {len(processed_df)} files")
        print_and_log(f"[+] DataFrame now has {len(processed_df.columns)} columns")
        print_and_log(f"Columns: {list(processed_df.columns)}")
        
    except Exception as e:
        print_and_log(f"[!] Error during processing: {e}", "error")
else:
    print_and_log("[!] No PDFs to process. Please add PDF files to the 'sample_pdfs' folder.", "error")


In [ ]:
processed_df.to_feather("/mnt/c/Users/WSTATION/Desktop/docling_mods/scripts/output/3_16_2026_198_recleaned_12.feather")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7: SAVE RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

if 'processed_df' in locals():
    try:
        processed_df.to_feather(OUTPUT_FEATHER)
        print_and_log(f"[+] Results saved to: {OUTPUT_FEATHER}")
        
        # Display summary
        print_and_log(f"\nProcessing Summary:")
        print_and_log(f"  - Input PDFs: {len(pdf_df)}")
        print_and_log(f"  - Successfully processed: {len(processed_df)}")
        print_and_log(f"  - Output columns: {len(processed_df.columns)}")
        print_and_log(f"  - Log file: {LOG_FILE}")
        
    except Exception as e:
        print_and_log(f"[!] Error saving results: {e}", "error")
else:
    print_and_log("[!] No processed data to save.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8: INSPECT RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

def inspect_results(feather_path=OUTPUT_FEATHER):
    """Load and display the processing results."""
    if not os.path.exists(feather_path):
        print_and_log(f"[!] Results file not found: {feather_path}")
        return None
        
    try:
        df = pd.read_feather(feather_path)
        print_and_log(f"[+] Loaded results: {len(df)} rows, {len(df.columns)} columns")
        print_and_log(f"\nColumns: {list(df.columns)}")
        print_and_log(f"\nFirst few rows:")
        display(df.head())
        return df
        
    except Exception as e:
        print_and_log(f"[!] Error loading results: {e}", "error")
        return None

# Run inspection if results exist
if os.path.exists(OUTPUT_FEATHER):
    results_df = inspect_results()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9: UTILITY - TEST SINGLE PDF
# ═══════════════════════════════════════════════════════════════════════════════

def test_single_pdf(pdf_path):
    """Test processing on a single PDF file."""
    if not os.path.exists(pdf_path):
        print_and_log(f"[!] PDF file not found: {pdf_path}")
        return
        
    print_and_log(f"[+] Testing single PDF: {os.path.basename(pdf_path)}")
    
    # Create single-row DataFrame
    test_df = pd.DataFrame({
        "PDFPath": [pdf_path],
        "FileName": [os.path.basename(pdf_path)]
    })
    
    try:
        result_df = do_docling_extraction(test_df)
        print_and_log(f"[+] Successfully processed!")
        print_and_log(f"Result columns: {list(result_df.columns)}")
        display(result_df.head())
        return result_df
        
    except Exception as e:
        print_and_log(f"[!] Error processing PDF: {e}", "error")
        return None

# Example usage (uncomment to test a specific file):
# test_single_pdf(os.path.join(PDF_INPUT_FOLDER, "your_pdf_file.pdf"))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10: QUICK PROCESSING FUNCTION (ALTERNATIVE TO RUNNING ALL CELLS)
# ═══════════════════════════════════════════════════════════════════════════════

def quick_process_folder(folder_path=None, output_path=None):
    """
    Quick function to process all PDFs in a folder.
    Args:
        folder_path (str, optional): Path to folder containing PDFs (defaults to sample_pdfs)
        output_path (str, optional): Output file path (defaults to output folder)
    """
    if folder_path is None:
        folder_path = PDF_INPUT_FOLDER
    
    if output_path is None:
        output_path = OUTPUT_FEATHER
    
    print_and_log(f"Processing PDFs from: {folder_path}")
    
    # Create DataFrame
    df = create_pdf_dataframe(folder_path)
    
    # Process with Docling
    processed_df = do_docling_extraction(df)
    
    # Save results
    processed_df.to_feather(output_path)
    
    print_and_log(f"Results saved to: {output_path}")
    return processed_df

# Example usage:
# results = quick_process_folder()

# Reconstruction text as markdown

The following code cells below are to reconstruct a PagesJson object as markdown. Markdown headers however are not accuarte (all of them are "##" ). this step is in preperation for adding provenence metadata for qdrant embedding

In [ ]:
# Cell 1: Setup
import pandas as pd
import json
 
df = pd.read_feather("/mnt/c/Users/WSTATION/Desktop/docling_mods/scripts/output/pages_json_table.feather")
row = df.iloc[4]
pages = json.loads(row['PagesJson'])
print(f"Total pages: {len(pages)}")
print(f"Original FullText length: {len(row['FullText'])} chars")

In [ ]:
# Cell 2: Markdown formatting functions
 
def table_to_markdown(table_data):
    """Convert a list-of-dicts table to a markdown table."""
    if not table_data or not isinstance(table_data, list):
        return "[Table]"
    
    if not isinstance(table_data[0], dict):
        return "[Table]"
    
    headers = list(table_data[0].keys())
    
    # Header row
    header_row = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    
    # Data rows
    data_rows = []
    for row_dict in table_data:
        cells = [str(row_dict.get(h, "")) for h in headers]
        data_rows.append("| " + " | ".join(cells) + " |")
    
    return "\n".join([header_row, separator] + data_rows)
 
 
def is_junk_item(text):
    """Filter out layout artifacts like lone colons, parens, asterisks."""
    if not text:
        return True
    stripped = text.strip()
    if len(stripped) <= 2 and not stripped.isalnum():
        return True
    return False
 
 
def items_to_markdown(items, skip_references=True):
    """Convert a page's items array to markdown string."""
    parts = []
    
    for item in items:
        if skip_references and item.get("is_reference"):
            continue
        
        label = item["label"]
        text = item.get("text", "")
        
        if label == "page_footer":
            continue
        
        if label == "section_header":
            if is_junk_item(text):
                continue
            parts.append(f"## {text.strip()}")
        
        elif label == "text":
            if is_junk_item(text):
                continue
            parts.append(text.strip())
        
        elif label == "list_item":
            if is_junk_item(text):
                continue
            parts.append(f"- {text.strip()}")
        
        elif label == "caption":
            if text.strip():
                parts.append(f"*{text.strip()}*")
        
        elif label == "formula":
            if text.strip():
                parts.append(f"$${text.strip()}$$")
        
        elif label == "table":
            if isinstance(text, list):
                parts.append(table_to_markdown(text))
            elif isinstance(text, str) and text.strip():
                parts.append(text.strip())
            else:
                parts.append("[Table]")
        
        elif label == "picture":
            parts.append("[Figure]")
        
        else:
            # Catch-all for unknown labels
            if isinstance(text, str) and text.strip() and not is_junk_item(text):
                parts.append(text.strip())
    
    return "\n\n".join(parts)

In [ ]:
# Cell 3: Reconstruct with markdown formatting
 
full_text_parts = []
for page in pages:
    if not page.get('items'):
        continue
    
    md = items_to_markdown(page['items'], skip_references=True)
    
    if md.strip():
        page_marker = f"\n\n<!-- PAGE {page['page_no']} -->\n\n"
        full_text_parts.append(page_marker + md)
 
reconstructed_md = "".join(full_text_parts).strip()
print(f"Reconstructed markdown length: {len(reconstructed_md)} chars")
print(reconstructed_md)

In [ ]:
# Cell 4: Show page structure
print("Page structure:")
print("-" * 60)
for page in pages:
    items = page.get('items', [])
    if not items:
        continue
    from collections import Counter
    labels = Counter(item['label'] for item in items if not item.get('is_reference'))
    print(f"Page {page['page_no']:2d}: {page['token_count_before_references']:4d} tokens | {dict(labels)}")

In [ ]:
# Cell 5: Compare original vs reconstructed (first section)
print("=" * 60)
print("ORIGINAL FullText (first 500 chars):")
print("=" * 60)
print(row['FullText'][:500])
print()
print("=" * 60)
print("RECONSTRUCTED markdown (first 500 chars):")
print("=" * 60)
print(reconstructed_md[:500])

In [ ]:
# Cell 6: Save
output_path = "/mnt/c/Users/WSTATION/Desktop/docling_mods/scripts/output/reconstructed_idx4.md"
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(reconstructed_md)
print(f"Saved to: {output_path}")

# Reconstructing text for topic modeling

the following tests are to reconstruct clean text by stripping everything but body text (attempts to remove authors, affiliations, encoding errors, etc.). the script version of these cells is `scripts/topic_text_reconstruction.py`

In [ ]:
# Cell 1: Setup
import pandas as pd
import json
import re
import unicodedata
 
df = pd.read_feather("/mnt/c/Users/WSTATION/Desktop/docling_mods/scripts/output/3_16_2026_198_recleaned.feather")
print(f"Loaded {len(df)} rows")

In [ ]:
# Cell 2: Core reconstruction function

# Labels to always skip
SKIP_LABELS = {"picture", "table", "caption", "formula", "page_footer", "footnote"}

DROP_SECTION_HEADERS = True
PAGE_FRONT_MATTER_MAX = 2  # filter front-matter on pages 1 and 2
PAGE1_CHARSPAN_MIN = 150
MIN_ITEM_WORDS = 3


def is_author_line(text):
    """Detect author listing lines."""
    if re.search(r'(\d,\d|\d\s*·|\d\s*,\s*\d)', text):
        if len(re.findall(r'[·&]', text)) >= 2:
            return True
    if len(re.findall(r'\s[·•]\s', text)) >= 2:
        return True
    if len(re.findall(r'/C1', text)) >= 2:
        if not re.match(r'(?i)^\s*keywords?\s', text):
            return True
    if len(re.findall(r'\s&\s', text)) >= 2:
        if not re.search(r'\b(is|are|was|were|has|have|the|this|that|from|with)\b', text.lower()):
            return True
    if len(re.findall(r'\s\.\s', text)) >= 2:
        if not re.search(r'\b(is|are|was|were|has|have|the|this|that|from|with)\b', text.lower()):
            return True
    if len(re.findall(r'Æ', text)) >= 1:
        return True
    return False


def is_affiliation_line(text):
    """Detect institutional affiliation lines by requiring affiliation-like start."""
    lower = text.lower().strip()

    starts_like_affiliation = bool(re.match(
        r'^(\d+\s+)?'
        r'('
        r'university|universit|institut|department|school of|'
        r'hospital|center for|centre for|college of|'
        r'laboratory|faculty of|division of|observatory|'
        r'research center|research centre|medical center|'
        r'medical centre|polytechnic|academy of|'
        r'[a-z]\.\s*(l\.\s*)?[a-z]'
        r')',
        lower
    ))
    if not starts_like_affiliation:
        return False

    if re.search(r'\b\d{4,6}\b', text):
        return True

    if len(text) < 200 and text.count(',') >= 2:
        return True

    return False


def is_junk_text(text):
    """Filter out layout artifacts: lone punctuation, single chars, etc."""
    if not text:
        return True
    stripped = text.strip()
    if len(stripped) <= 3:
        return True
    if not re.search(r'[a-zA-Z]', stripped):
        return True
    if re.match(r'^\s*\S+@\S+\s*$', stripped):
        return True
    if re.match(r'^\s*(https?://|doi:|www\.)\S+\s*$', stripped, re.I):
        return True
    return False


def is_boilerplate(text):
    """Catch common boilerplate that survives label filtering."""
    lower = text.lower().strip()
    patterns = [
        r'^received:?\s+\d',
        r'^accepted:?\s+\d',
        r'^published\s+online',
        r'^available\s+online',
        r'^\(?c\)?\s*\d{4}',
        r'^©',
        r'^the\s+author\(?s?\)?',
        r'^this\s+(article|work)\s+is\s+(published|distributed|licensed)',
        r'^open\s+access',
        r'^electronic\s+supplementary\s+material',
        r'^keywords?\s',
        r'^abbreviations?\s',
        r'^e-?mail:',
        r'^corresponding\s+author',
        r'^conflict\s+of\s+interest',
        r'^data\s+availability',
        r'^funding',
        r'^acknowledgment',
        r'^author\s+contribution',
        r'^supplementary\s+(data|material|information)',
    ]
    return any(re.match(p, lower) for p in patterns)


def classify_item(item, page_no):
    """Return (kept: bool, reason: str) for a single item."""
    label = item.get("label", "")
    text = item.get("text", "")
    if not isinstance(text, str):
        text = ""

    if item.get("is_reference", False):
        return False, "is_reference"
    if label in SKIP_LABELS:
        return False, f"label={label}"
    if DROP_SECTION_HEADERS and label == "section_header":
        return False, "section_header"
    if is_junk_text(text):
        return False, "junk"

    if page_no <= PAGE_FRONT_MATTER_MAX:
        if is_author_line(text):
            return False, "author_line"
        if is_affiliation_line(text):
            return False, "affiliation"
        charspan = item.get("charspan", [0, 0])
        span_len = charspan[1] - charspan[0] if len(charspan) == 2 else len(text)
        if span_len < PAGE1_CHARSPAN_MIN:
            return False, f"charspan={span_len}<{PAGE1_CHARSPAN_MIN}"

    if is_boilerplate(text):
        return False, "boilerplate"

    word_count = len(re.findall(r'[a-zA-Z]{2,}', text))
    if word_count < MIN_ITEM_WORDS:
        return False, f"words={word_count}<{MIN_ITEM_WORDS}"

    return True, ""


def extract_topic_text(pages_json_str):
    """
    Reconstruct clean body text from PagesJson for topic modeling.
    No markdown, no tables, no figures, no references, no front-matter.
    """
    if not pages_json_str or pages_json_str == "ANALYSIS_ERROR":
        return ""

    try:
        pages = json.loads(pages_json_str)
    except (json.JSONDecodeError, TypeError):
        return ""

    body_parts = []

    for page in pages:
        if page is None:
            continue

        items = page.get("items", [])
        page_no = page.get("page_no", 0)

        for item in items:
            kept, _ = classify_item(item, page_no)
            if kept:
                text = item.get("text", "")
                if isinstance(text, str):
                    body_parts.append(text.strip())

    return "\n\n".join(body_parts)

In [ ]:
# Cell 3: Test on all rows
 
df["TopicText"] = df["PagesJson"].apply(extract_topic_text)
 
for idx, row in df.iterrows():
    orig_len = len(row["FullText"]) if isinstance(row["FullText"], str) else 0
    topic_len = len(row["TopicText"])
    orig_words = len(row["FullText"].split()) if isinstance(row["FullText"], str) else 0
    topic_words = len(row["TopicText"].split()) if row["TopicText"] else 0
    reduction = (1 - topic_len / orig_len) * 100 if orig_len > 0 else 0
    print(f"Row {idx:2d}: FullText={orig_words:6d} words | TopicText={topic_words:6d} words | {reduction:.1f}% reduction")

In [ ]:
# Cell 4: Inspect a specific row side by side
 
ROW = 0  # change as needed
row = df.iloc[ROW]
 
print("=" * 60)
print(f"ROW {ROW}: First 1500 chars of TopicText")
print("=" * 60)
print(row["TopicText"][:1500])

In [ ]:
# Cell 5: Show what got dropped from page 1
 
pages = json.loads(df.iloc[ROW]["PagesJson"])
for pg in pages:
    if pg is None or not pg.get("items"):
        continue
    page_no = pg.get("page_no", 0)
    if page_no < 1 or page_no > 2:
        continue

    print(f"\nPage {page_no} items - KEPT vs DROPPED:")
    print("-" * 60)
    for item in pg.get("items", []):


        label = item["label"]
        text = item.get("text", "")
        if not isinstance(text, str):
            text = f"[{type(text).__name__}]"
        text_preview = text[:300]
    
        # replicate the filtering logic
        kept = True
        reason = ""
    
        if item.get("is_reference"):
            kept, reason = False, "is_reference"
        elif label in SKIP_LABELS:
            kept, reason = False, f"label={label}"
        elif DROP_SECTION_HEADERS and label == "section_header":
            kept, reason = False, "section_header"
        elif is_junk_text(text):
            kept, reason = False, "junk"
        elif page_no <= PAGE_FRONT_MATTER_MAX:
            if is_author_line(text):
                kept, reason = False, "author_line"
            elif is_affiliation_line(text):
                kept, reason = False, "affiliation"
            else:
                charspan = item.get("charspan", [0, 0])
                span_len = charspan[1] - charspan[0] if len(charspan) == 2 else len(text)
                if span_len < PAGE1_CHARSPAN_MIN:
                    kept, reason = False, f"charspan={span_len}<{PAGE1_CHARSPAN_MIN}"
                
        if kept and is_boilerplate(text):
            kept, reason = False, "boilerplate"
        if kept:
            word_count = len(re.findall(r'[a-zA-Z]{2,}', text))
            if word_count < MIN_ITEM_WORDS:
                kept, reason = False, f"words={word_count}<{MIN_ITEM_WORDS}"
    
        charspan = item.get("charspan", [0, 0])
        span_len = charspan[1] - charspan[0] if len(charspan) == 2 else len(text)
        status = "KEPT" if kept else f"DROP ({reason})"
        print(f"  [{status:30s}] cs={span_len:4d}  {label:20s} {text_preview!r}")

In [ ]:
# Cell 6: Export all rows page 1+2 diagnostics + TopicText preview to file
# Removed page1 = pages[1] if len(pages) > 1 and pages[1].get("items") else pages[0],
# replaced with the for pg in pages: loop filtering to pages 1 and 2
# Changed page1.get("page_no", 0) <= 1 to page_no <= PAGE_FRONT_MATTER_MAX
# Changed for item in page1.get("items", []): to for item in pg.get("items", []):
output_path = "/mnt/c/Users/WSTATION/Desktop/docling_mods/topic_text_diagnostics4.txt"
with open(output_path, 'w', encoding='utf-8') as f:
    for row_idx in range(len(df)):
        row = df.iloc[row_idx]
        pages_str = row.get("PagesJson", "")
        if not pages_str or pages_str == "ANALYSIS_ERROR":
            f.write(f"row {row_idx}\n\nSKIPPED (no PagesJson)\n\n{'='*60}\n\n")
            continue
        pages = json.loads(pages_str)
        f.write(f"row {row_idx}\n\n")
        for pg in pages:
            if pg is None or not pg.get("items"):
                continue
            page_no = pg.get("page_no", 0)
            if page_no < 1 or page_no > 2:
                continue
            f.write(f"Page {page_no} items - KEPT vs DROPPED:\n")
            f.write("-" * 60 + "\n")
            for item in pg.get("items", []):
                label = item["label"]
                text = item.get("text", "")
                if not isinstance(text, str):
                    text = f"[{type(text).__name__}]"
                text_preview = text[:500]
                kept = True
                reason = ""
                if item.get("is_reference"):
                    kept, reason = False, "is_reference"
                elif label in SKIP_LABELS:
                    kept, reason = False, f"label={label}"
                elif DROP_SECTION_HEADERS and label == "section_header":
                    kept, reason = False, "section_header"
                elif is_junk_text(text):
                    kept, reason = False, "junk"
                elif page_no <= PAGE_FRONT_MATTER_MAX:
                    if is_author_line(text):
                        kept, reason = False, "author_line"
                    elif is_affiliation_line(text):
                        kept, reason = False, "affiliation"
                    else:
                        charspan = item.get("charspan", [0, 0])
                        span_len = charspan[1] - charspan[0] if len(charspan) == 2 else len(text)
                        if span_len < PAGE1_CHARSPAN_MIN:
                            kept, reason = False, f"charspan={span_len}<{PAGE1_CHARSPAN_MIN}"
                if kept and is_boilerplate(text):
                    kept, reason = False, "boilerplate"
                if kept:
                    word_count = len(re.findall(r'[a-zA-Z]{2,}', text))
                    if word_count < MIN_ITEM_WORDS:
                        kept, reason = False, f"words={word_count}<{MIN_ITEM_WORDS}"
                charspan = item.get("charspan", [0, 0])
                span_len = charspan[1] - charspan[0] if len(charspan) == 2 else len(text)
                status = "KEPT" if kept else f"DROP ({reason})"
                f.write(f"  [{status:30s}] cs={span_len:4d}  {label:20s} {text_preview!r}\n")
            f.write("\n")
        f.write(f"{'='*60}\n")
        f.write(f"ROW {row_idx}: First 1500 chars of TopicText\n")
        f.write(f"{'='*60}\n")
        f.write((row["TopicText"] or "")[:1500])
        f.write(f"\n\n{'='*60}\n\n")
print(f"Wrote diagnostics for {len(df)} rows to {output_path}")

In [ ]:
import pandas as pd

glyphs = pd.read_feather('/mnt/c/Users/WSTATION/Desktop/docling_mods/scripts/output/3_16_2026_glyph_unifb_subset.feather')